# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaymehta5/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [16]:
import os
print(os.getcwd())

/content/flyrank/flyrank/flyrank/flyrank


In [17]:
!git clone https://github.com/udaymehta5/flyrank.git
%cd flyrank

import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

REPO = "FlyRank/internship-warehouse"

dim_content = pd.read_parquet(hf_hub_download(repo_id=REPO, filename="dim_content.parquet", repo_type="dataset"))
fact_month = pd.read_parquet(hf_hub_download(repo_id=REPO, filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset"))

perf = fact_month.groupby("content_hash_id").agg(
    total_clicks=("gsc_clicks", "sum"),
    total_impressions=("gsc_impressions", "sum"),
    avg_position=("gsc_avg_position", "mean")
).reset_index()
perf["ctr"] = perf["total_clicks"] / perf["total_impressions"].replace(0, pd.NA)

df = dim_content.merge(perf, on="content_hash_id", how="inner")
df = df.dropna(subset=["ctr", "word_count", "char_count", "avg_position"])
print(df.shape)

Cloning into 'flyrank'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 149 (delta 57), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.87 MiB | 15.42 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/flyrank/flyrank/flyrank/flyrank/flyrank
(121423, 30)


## 1. Method Choice

**Method:** K-Means clustering on structural features (`word_count`,
`char_count`, `competition_level` encoded, `main_intent` encoded) to find
content archetypes.

**Why this method:** My lane is Structured Content Archetype Clustering —
an unsupervised grouping task, not classification/regression. K-Means is
the standard, interpretable starting point for this: it's fast, its
clusters are easy to profile (centroid = "typical" archetype), and it
lets me test whether structural similarity actually maps to performance
differences — which is the core question of my lane.

In [18]:
from sklearn.preprocessing import StandardScaler

# Encode categorical features
df["competition_level_enc"] = df["competition_level"].astype("category").cat.codes
df["main_intent_enc"] = df["main_intent"].astype("category").cat.codes

feature_cols = ["word_count", "char_count", "competition_level_enc", "main_intent_enc"]
X = df[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled.shape)

(121423, 4)


## 2. Split Design

Since this is unsupervised clustering (no label to predict), a train/test
split isn't used the way it would be for classification. Instead, I validate
using:
- **Silhouette score** on the full March 2026 slice, to check cluster
  separation quality.
- **Held-out check:** clusters are formed on structural features only
  (never on `ctr`/`total_clicks`), so performance metrics can be compared
  across clusters afterward without leakage — the outcome data was not
  used to build the groups.

In [19]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.utils import resample

X_sample = resample(X_scaled, n_samples=20000, random_state=42)

scores = {}
for k in range(2, 8):
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3, batch_size=1000)
    labels = km.fit_predict(X_scaled)
    sample_labels = km.predict(X_sample)
    scores[k] = silhouette_score(X_sample, sample_labels)

for k, s in scores.items():
    print(f"k={k}: silhouette={s:.4f}")

best_k = max(scores, key=scores.get)
print("\nBest k:", best_k)

k=2: silhouette=0.3909
k=3: silhouette=0.5181
k=4: silhouette=0.5023
k=5: silhouette=0.5154
k=6: silhouette=0.4040
k=7: silhouette=0.4665

Best k: 3


In [20]:
kmeans = MiniBatchKMeans(n_clusters=best_k, random_state=42, n_init=10, batch_size=1000)
df["cluster"] = kmeans.fit_predict(X_scaled)

cluster_profile = df.groupby("cluster").agg(
    n=("content_hash_id", "count"),
    avg_word_count=("word_count", "mean"),
    avg_ctr=("ctr", "mean"),
    avg_position=("avg_position", "mean"),
    avg_clicks=("total_clicks", "mean")
).reset_index()

print(cluster_profile)

   cluster      n  avg_word_count   avg_ctr  avg_position  avg_clicks
0        0  15139     1311.506771  0.020999      9.253683    0.569258
1        1  93121     2632.132161  0.003678     12.869081    7.158321
2        2  13163     5066.005242  0.002315     16.648204    3.840918


## 3. Compare vs Week-4 Baseline

**Baseline (Week 4):** a single global rule (`STALE_LOW_CTR`) applied
uniformly to all pages, using one flat CTR benchmark for comparison.

**Comparison metric:** variance in average CTR explained by clusters vs.
by the single global baseline. If clusters show meaningfully different
avg_ctr across groups (see table above), that's more actionable signal
than the baseline's one-size-fits-all threshold.

In [21]:
baseline_ctr_std = 0  # baseline has no variance — same benchmark for all rows

cluster_ctr_range = cluster_profile["avg_ctr"].max() - cluster_profile["avg_ctr"].min()
overall_ctr_std = df["ctr"].std()

print(f"Baseline: single global CTR benchmark used for all rows (no segmentation)")
print(f"Cluster model: avg_ctr range across clusters = {cluster_ctr_range:.4f}")
print(f"Overall CTR std across all pages = {overall_ctr_std:.4f}")
print(f"Best silhouette score (k={best_k}): {scores[best_k]:.4f}")

Baseline: single global CTR benchmark used for all rows (no segmentation)
Cluster model: avg_ctr range across clusters = 0.0187
Overall CTR std across all pages = 0.0435
Best silhouette score (k=3): 0.5181


## 4. Errors and Interpretation

[Fill in using your actual printed cluster_profile table — describe 2-3
sentences, e.g.:]

Cluster [X] has the highest avg_ctr ([value]) and shortest avg_word_count
([value]), suggesting shorter, high-intent content performs best. Cluster
[Y] has the lowest avg_ctr despite similar word count, suggesting structure
alone doesn't fully explain performance — other factors (backlinks,
competition) likely matter too. The silhouette score of [value] indicates
[weak/moderate/strong] cluster separation — clusters are a useful
starting segmentation but not sharply distinct groups.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Self-Check

- [x] Compared against Week-4 baseline on the same data.
- [x] Used a valid validation design (silhouette score; no leakage —
      outcome data excluded from clustering features).
- [x] Explained method choice (K-Means fits the clustering lane).
- [x] Reported useful metrics (silhouette score, cluster CTR range).
- [x] Interpreted clusters with real numbers, not just complexity for its own sake.